# scVI (RNA-only) benchmark — Patient 1 (03H096 / PB2)

**Role in the paper:** Cross-modal benchmark (Extended Data Fig. 7). It asks how much
of the trimodal structure — in particular the Root/CSC cluster — can be
recovered from transcriptome alone.

**What this notebook does**
1. Loads the raw MuData and keeps the barcodes retained by MultiVI
2. Runs the same RNA quality control as the MultiVI notebook
3. Selects 4000 highly variable genes and trains **scVI**
4. Builds neighbours, Leiden clusters and a ForceAtlas2 layout
5. Compares the scVI clusters with the trimodal `Cluster_Final` labels and
   highlights the Root/CSC cluster
6. Writes `ScVI_PB2.h5ad`, consumed by `04_Single_modality_checks/LSC_recovery_Patient1_03H096.ipynb`

**Objects**
- **Reads:** `DATA_DIR / "Teaseq_PB2.h5mu"` and
  `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad"`
- **Creates:** `DATA_DIR / "04_Single_modality_checks/RNA/ScVI_PB2.h5ad"`

## Paths and settings

In [ ]:
from pathlib import Path

# Root of the companion data package. Point this at your local copy.
DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "outputs/scVI_PB2"     # figures and tables written by this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os

import anndata as ad
import matplotlib.pyplot as plt
import mudata as md
import muon
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

## Load the raw MuData and the cleaned MultiVI reference

In [ ]:
### Read the H5 file ###

adataPB2 = muon.read(DATA_DIR / "Teaseq_PB2.h5mu")
adataPB2.var_names_make_unique()

In [ ]:
# Cleaned MultiVI object: source of the barcode list and cluster labels.
adataMultiPB2 = ad.read_h5ad(
    DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad"
)

In [ ]:
# Normalise the antibody names to the ADT.anti.hu.<clone> convention.
adataPB2.mod['protein'].var.index = 'ADT.anti.hu.' + adataPB2.mod['protein'].var.index.str.split('-').str[0]

In [ ]:
### Annotation of all the samples ###

adataPB2.obs.index = [name + '_PB2' for name in adataPB2.obs_names]

In [ ]:
adata = adataPB2

## Split modalities and restrict to the QC-passing barcodes

In [ ]:
### Extracting RNA data only ###

rna_adata = adata.mod['rna']
prot_adata = adata.mod['protein']
atac_adata = adata.mod['atac']
rna_adata.obs.index = adata.obs.index
prot_adata.obs.index = adata.obs.index
atac_adata.obs.index = adata.obs.index

In [ ]:
# Barcodes retained by MultiVI; every modality is subset to this list.
cell_ids_list = adataMultiPB2.obs_names.tolist()
print(len(cell_ids_list), 'cells;', cell_ids_list[:3])

In [ ]:
# Filter the MuData object
rna_adata = rna_adata[rna_adata.obs.index.isin(cell_ids_list)]
rna_adata

In [ ]:
# Filter the MuData object
prot_adata = prot_adata[prot_adata.obs.index.isin(cell_ids_list)]
prot_adata

## RNA quality control

In [ ]:
rna_adata.obs['Cluster_Final']=adataMultiPB2.obs['Cluster_Final']

In [ ]:
### Top ranking expressed gene ###

sc.pl.highest_expr_genes(rna_adata, n_top=20, )

In [ ]:
### Three plot summary BEFORE filtering ###

rna_adata.var['mt'] = rna_adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(rna_adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

rna_adata.var_names_make_unique()

sc.pl.violin(rna_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

rna_adata

In [ ]:
### Summary of genes and counts BEFORE filtering ###

sc.pl.scatter(rna_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(rna_adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# RNA cell/feature filters as reported in Methods. Most cells already
# pass them because the barcode list was restricted above.

sc.pp.filter_cells(rna_adata, min_genes=200)
sc.pp.filter_genes(rna_adata, min_cells=3)
rna_adata = rna_adata[rna_adata.obs.n_genes_by_counts < 5000, :]
rna_adata = rna_adata[rna_adata.obs.n_genes_by_counts > 350, :]
rna_adata = rna_adata[rna_adata.obs.pct_counts_mt < 40, :]

In [ ]:
### Three plot summary AFTER filtering ###

sc.pl.violin(rna_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

rna_adata

In [ ]:
### Summary of genes and counts AFTER filtering ###

sc.pl.scatter(rna_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(rna_adata, x='total_counts', y='n_genes_by_counts')

## Keep the informative antibodies

The protein matrix travels with the RNA object so the two benchmarks share exactly the same cells; it is not used for training.

In [ ]:
# Filter out variables that contain 'IgG' in their names
prot_adata = prot_adata[:, ~prot_adata.var.index.str.contains('Rat|Mouse|Hamster')]
prot_adata

## Assemble the AnnData / MuData for scVI

In [ ]:
### Add the protein information in the Anndata object ###

adata=rna_adata

# Get the cell names of the filtered RNA data
cell_names = rna_adata.obs_names

# Add the protein expression data to the obsm attribute of the RNA AnnData object
rna_adata.obsm['protein_expression'] = prot_adata.X

# Add the names of the protein markers to the uns attribute of the RNA AnnData object
rna_adata.uns['protein_names'] = prot_adata.var_names

adata=rna_adata

adata

In [ ]:
### Normalization of the data ###

adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obs_names_make_unique()

In [ ]:
### Remergeed the protein data with the rna data ###

# Create a new AnnData object for the protein expression data
protein_adata = ad.AnnData(adata.obsm["protein_expression"])

# Set the obs_names of the protein AnnData object to match the RNA AnnData object
protein_adata.obs_names = adata.obs_names

# Set the var_names of the protein AnnData object to the names of the protein markers
protein_adata.var_names = adata.uns['protein_names']

# Remove the protein expression data from the obsm attribute of the RNA AnnData object
del adata.obsm["protein_expression"]

# Create a MuData object containing both the RNA and protein data
mdata = md.MuData({"rna": adata, "protein": protein_adata})

# Extract the batch information from the cell identifiers
batches = [cell_id.split('_')[-1] for cell_id in mdata.obs_names]

# Add the batch information to the rna modality
mdata.mod['rna'].obs['batch'] = pd.Categorical(batches)

mdata

In [ ]:
### Select the high variable gene ###

sc.pp.highly_variable_genes(
    mdata.mod["rna"],
    n_top_genes=4000,
    flavor="seurat_v3",
    batch_key="batch",
    layer="counts",
)
# Place subsetted counts in a new modality
mdata.mod["rna_subset"] = mdata.mod["rna"][
    :, mdata.mod["rna"].var["highly_variable"]
].copy()
mdata.update()
mdata

In [ ]:
### Check for duplicates ###

for modality in mdata.mod:
    print(f"Checking modality: {modality}")
    df = mdata.mod[modality].to_df()
    if len(df.columns) != len(df.columns.unique()):
        print(f"There are duplicate column names in modality {modality}")
        duplicates = df.columns[df.columns.duplicated()].unique()
        print(f"The following columns are duplicated: {duplicates}")
    else:
        print(f"There are no duplicate column names in modality {modality}")

## Train scVI on the highly variable genes

In [ ]:
mdata=mdata.mod['rna_subset'].copy()

In [ ]:
scvi.model.SCVI.setup_anndata(
    mdata,
    layer="counts",
)

In [ ]:
model = scvi.model.SCVI(mdata)

In [ ]:
model.train(
    check_val_every_n_epoch=1,
    max_epochs=400,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_monitor="elbo_validation",
)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"

latent = model.get_latent_representation()
mdata.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
# Ensure convergence
train_test_results = model.history["elbo_train"]
train_test_results["elbo_validation"] = model.history["elbo_validation"]
train_test_results.iloc[10:].plot(logy=True)  # exclude first 10 epochs
plt.show()

## Compare the scVI clusters with the trimodal labels

In [ ]:
adata = ad.read_h5ad(DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad")
adata

In [ ]:
muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color=["Cluster_Final"],
    frameon=False,
    ncols=1,
)

In [ ]:
### Compute clusters and visualize the latent space ###

sc.pp.neighbors(mdata, use_rep="X_scVI")
sc.tl.umap(mdata)
sc.tl.leiden(mdata, resolution= 1 ,key_added="leiden_scVI")
sc.tl.draw_graph(mdata, layout='fa')
mdata

In [ ]:
### Plot the ForceAtlas2 layout coloured by scVI cluster ###

muon.pl.embedding(
    mdata,
    basis="X_draw_graph_fa",
    color=["leiden_scVI"],
    frameon=False,
    ncols=1,
)

In [ ]:
mdata.obs['Cluster_Final']=adata.obs['Cluster_Final']

In [ ]:
### scVI clusters next to the trimodal Cluster_Final labels ###

muon.pl.embedding(
    mdata,
    basis="X_draw_graph_fa",
    color=["leiden_scVI", "Cluster_Final"],
    frameon=False,
    ncols=2,
)

## Genes detected per scVI cluster (ANOVA + Tukey HSD)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import f_oneway
from scipy.stats import f as fdist
from statsmodels.stats.multicomp import pairwise_tukeyhsd


def _format_p_from_logsf(logp):
    """Format p-value from log(p)."""
    if not np.isfinite(logp):
        return "p=NA", None

    neglog10p = (-logp) / np.log(10)

    if neglog10p > 300:
        return f"p < 1e-{int(np.floor(neglog10p))}", neglog10p
    else:
        p_val = np.exp(logp)
        return f"p={p_val:.2e}", neglog10p


def plot_leiden_scVI_n_genes(mdata, highlight_max=True):
    """
    Bar plot (mean + 95% CI) per leiden_scVI for n_genes.
    Uses colors from mdata.uns['leiden_scVI_colors'] if available.
    Also prints ANOVA and Tukey HSD results.
    """

    # Checks
    if 'leiden_scVI' not in mdata.obs:
        raise KeyError("`leiden_scVI` missing from mdata.obs")

    if 'n_genes' not in mdata.obs:
        raise KeyError("`n_genes` missing from mdata.obs")

    # Ensure categorical
    cl = mdata.obs['leiden_scVI']
    if not pd.api.types.is_categorical_dtype(cl):
        mdata.obs['leiden_scVI'] = cl.astype('category')
        cl = mdata.obs['leiden_scVI']

    categories = cl.cat.categories

    raw_colors = mdata.uns.get('leiden_scVI_colors', None)

    if raw_colors is None:
        # fallback: simple gray palette
        colors = ["gray"] * len(categories)
    else:
        colors = list(raw_colors)

        if len(colors) > len(categories):
            colors = colors[:len(categories)]

        if len(colors) < len(categories):
            raise ValueError(
                f"Not enough colors: {len(colors)} for {len(categories)} clusters"
            )

    color_map = {cat: col for cat, col in zip(categories, colors)}

    # Data
    df = mdata.obs[['leiden_scVI', 'n_genes']].copy()
    df['n_genes'] = pd.to_numeric(df['n_genes'], errors='coerce')
    df = df.dropna(subset=['leiden_scVI', 'n_genes'])

    df['leiden_scVI'] = df['leiden_scVI'].astype('category')
    df['leiden_scVI'] = df['leiden_scVI'].cat.set_categories(categories, ordered=True)

    # Stats
    means  = df.groupby('leiden_scVI', observed=True)['n_genes'].mean().reindex(categories)
    counts = df.groupby('leiden_scVI', observed=True)['n_genes'].count().reindex(categories)
    stds   = df.groupby('leiden_scVI', observed=True)['n_genes'].std().reindex(categories)

    sem  = stds / np.sqrt(counts)
    ci95 = (1.96 * sem).fillna(0.0)

    # ANOVA
    groups = [df.loc[df['leiden_scVI'] == c, 'n_genes'].values for c in categories]
    groups_anova = [g for g in groups if len(g) >= 2]

    k = len(groups_anova)
    N = sum(len(g) for g in groups_anova)

    dfn = k - 1
    dfd = N - k

    if k >= 2:
        F, _ = f_oneway(*groups_anova)
        logp = fdist.logsf(F, dfn, dfd)
        p_text, neglog10p = _format_p_from_logsf(logp)
        anova_title = f"ANOVA F({dfn},{dfd})={F:.2f}, {p_text}"
    else:
        F = np.nan
        anova_title = "ANOVA not applicable"

    # Tukey
    tukey = None
    try:
        if df['leiden_scVI'].nunique() >= 2:
            tukey = pairwise_tukeyhsd(df['n_genes'].values, df['leiden_scVI'].values)
    except Exception:
        tukey = None

    # Plot
    x = np.arange(len(categories))

    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(
        x,
        means.values,
        yerr=ci95.values,
        color=[color_map[c] for c in categories],
        edgecolor="black",
        linewidth=1.2,
        alpha=0.9,
        capsize=5
    )

    if highlight_max and np.isfinite(means.values).any():
        max_idx = int(np.nanargmax(means.values))
        bars[max_idx].set_hatch("//")
        bars[max_idx].set_linewidth(2.0)

    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in categories], rotation=0)

    ax.set_xlabel("leiden_scVI")
    ax.set_ylabel("Average n_genes")
    ax.set_title(f"Average n_genes per leiden_scVI\n({anova_title})")

    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

    # Console output
    print("\n[Summary stats] n_genes")
    summary = pd.DataFrame({
        "Mean": means,
        "Count": counts,
        "Std": stds,
        "SEM": sem,
        "CI95_Lower": means - ci95,
        "CI95_Upper": means + ci95
    })
    print(summary)

    if np.isfinite(F):
        print(f"\n[ANOVA] F({dfn},{dfd}) = {F:.6g}")
        print(f"[ANOVA] {p_text}")
        if 'neglog10p' in locals() and neglog10p is not None:
            print(f"[ANOVA] -log10(p) ≈ {neglog10p:.2f}")
    else:
        print("\n[ANOVA] Not available")

    if tukey is not None:
        print("\n[Tukey HSD results]")
        tukey_df = pd.DataFrame(
            tukey._results_table.data[1:],
            columns=tukey._results_table.data[0]
        )
        print(tukey_df.to_string(index=False))
    else:
        print("\n[Tukey HSD] Not available")


# Usage
plot_leiden_scVI_n_genes(mdata)

## Highlight the Root/CSC cluster

In [ ]:
adata=mdata.copy()

In [ ]:
cluster_to_highlight = "6"

import numpy as np
import pandas as pd

adata.obs["highlight"] = np.where(
    adata.obs["Cluster_Final"] == cluster_to_highlight,
    cluster_to_highlight,
    "Other"
)

# Draw the highlighted cluster on top of the grey background.
adata.obs["highlight"] = pd.Categorical(
    adata.obs["highlight"],
    categories=["Other", cluster_to_highlight],
    ordered=True
)

adata.uns["highlight_colors"] = [
    "#D3D3D3",  # Other → grey
    "#FF0000"   # Highlight → red
]

muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color="highlight",
    size=50,
    frameon=False,
)

## Save the RNA embedding

In [ ]:
import os
import pandas as pd

# `protein_names` cannot be serialised to h5ad, drop it first.
if "protein_names" in mdata.uns:
    del mdata.uns["protein_names"]

target_dir = DATA_DIR / "04_Single_modality_checks/RNA"
target_dir.mkdir(parents=True, exist_ok=True)

output_path = target_dir / "ScVI_PB2.h5ad"
mdata.write(output_path)

print("Saved:", output_path, "|", output_path.is_file())